# Expand AATC

In https://drivenbydata.atlassian.net/browse/DD-2117 it is requested that the AATC is expanded, so that AAT concepts submitted by PAN, do not fail, if the submitted AAT concepts are valid concepts (gpv:Concept and not gpv:GuideTerm)


## How large will AATC get?

If, we include:
* concepts with EN, and no NL labels
* obsolete concepts

## development steps

- [x] create test data - because the SPARQL query in [aatc_generate.py](aatc_generate.py) is a bit finicky, and the amount of return data is large, I would like to start by assembling test data. Containing: 1 concept with only @en label, 1 concept with @nl and @en labels and one concept with @en-US label 
* [ ] write select query: select all concepts with @en or @en-US labels and include also @nl labels when existing
* write construct query
* test construct results
* run construct query against getty SPARQL end-point
* count number of resulting triples
* test results


In [72]:
#### boiler plate functions ####
# imports SPARQL prefixes and functions defs

from pprint import pprint
from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 
from rdflib import Graph

prefixes = '''    
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX aat: <http://vocab.getty.edu/aat/>
PREFIX gvp: <http://vocab.getty.edu/ontology#> 
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX skosxl: <http://www.w3.org/2008/05/skos-xl#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX aatc: <http://vocabularies.dans.knaw.nl/aatconcepts/>
'''    


def sparql_endpoint_query(query, format, endpoint="http://vocab.getty.edu/sparql"):
    formats = {"json": JSON, "turtle": TURTLE, "csv": CSV}
    f_ = formats[format]
    endpoint = endpoint 
    sparql = SPARQLWrapper(endpoint)

    query = prefixes + query     
    sparql.setQuery(query)

    sparql.setReturnFormat(f_)
    results = sparql.query().convert()    
    # sparql.setReturnFormat(XML)
    # results = sparql.query()
    return results
    # # Print the results
    # print("Subject ID: ", subjectID)
    # 

def print_sparql_results(results):
    for row in results["results"]["bindings"]:
        return (row)

def sparql_file_query(query, format, filepath):
    g = Graph()
    g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results


In [91]:
#### CONSTRUCT query to create test data test/sample-data.ttl ####
# including the concept & all its skosxl:prefLabel nodes, 
# as well as the skosxl:prefLabel nodes' property value pairs

# @en http://vocab.getty.edu/aat/300189559 and @nl
# @en-US http://vocab.getty.edu/aat/300022441 and @nl (no @en label)
# @en only no Dutch http://vocab.getty.edu/aat/300456698'

query_construct = '''
CONSTRUCT
{     ?concept a gvp:Concept ;
               skos:inScheme aat: ;
               skosxl:prefLabel ?preflabel ;
               gvp:prefLabelGVP ?gpvlabel .

      ?preflabel skosxl:literalForm  ?preflabelVal .
      ?gpvlabel skosxl:literalForm  ?gpvlabelVal .
} 
WHERE 
{
   VALUES ?concept { aat:300189559 aat:300022441 aat:300456698 aat:300430847 } 

   ?concept a gvp:Concept .
   ?concept skosxl:prefLabel ?preflabel.
   ?concept gvp:prefLabelGVP ?gpvlabel .
   ?preflabel skosxl:literalForm  ?preflabelVal .
   ?gpvlabel skosxl:literalForm  ?gpvlabelVal .      
}
ORDER BY ?concept
'''
testq = sparql_endpoint_query(query=query_construct, format='turtle')
with open('tests/sample-data.ttl', 'wb') as sampledata:
      sampledata.write(testq)


In [74]:
#### QUERY Test Data #### 

# query concepts with @en labels - display labels
# expected query results: 
# http://vocab.getty.edu/aat/300189559 male
# http://vocab.getty.edu/aat/300456698 Nestor notabilis (species)

query_test = '''
SELECT  ?concept ?label_literal_en
WHERE
{
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel .
        

     ?preflabel  dcterms:language aat:300388277 ; skosxl:literalForm ?label_literal_en.   # lang: @en
}
'''
results = sparql_file_query(query=query_test, format='ttl', filepath='tests/sample-data.ttl')
for row in results:
    print(row.concept, row.label_literal_en)


In [75]:
#### QUERY Test Data #### 

# query concepts with @en @en-US labels - display labels in both @en and @nl - including concepts without @nl labels
# expected query results: 
# http://vocab.getty.edu/aat/300189559 male None
# http://vocab.getty.edu/aat/300189559 None mannelijk
# http://vocab.getty.edu/aat/300022441 colored pencils None
# http://vocab.getty.edu/aat/300022441 None kleurpotloden
# http://vocab.getty.edu/aat/300456698 Nestor notabilis (species) None

query_test = '''
SELECT  ?concept ?label_literal_en ?label_literal_nl
WHERE
{
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel .
        
    { ?preflabel  dcterms:language aat:300388277 ; skosxl:literalForm ?label_literal_en. }
    UNION    
    { ?preflabel  dcterms:language aat:300387822 ; skosxl:literalForm ?label_literal_en_us . 
      BIND (STRLANG(STR(?label_literal_en_us), 'en') AS  ?label_literal_en)} # lang: @en-US - converts to @en
    UNION
    { ?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . }
}
'''
results = sparql_file_query(query=query_test, format='ttl', filepath='tests/sample-data.ttl')
for row in results:
    print(row.concept, row.label_literal_en, row.label_literal_nl)


In [ ]:
### CONSTRUCT query for test data ### 
# where concept has skos:prefLabel in @en and @nl (if the source contains @nl)

query__construct_test = '''
CONSTRUCT {
    <http://vocabularies.dans.knaw.nl/aatconcepts> a skos:ConceptScheme .

    ?concept a skos:Concept ;
        skos:inScheme aat: ;
        skos:inScheme <http://vocabularies.dans.knaw.nl/aatconcepts> ;
        skos:prefLabel ?label_literal_en ;
        skos:prefLabel ?label_literal_nl ;
        dcterms:issued ?issuedDate .

        ?label_literal_en dcterms:language 
}

WHERE
{
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel .
        
    { ?preflabel  dcterms:language aat:300388277 ; skosxl:literalForm ?label_literal_en. 
       }
    UNION    
    { ?preflabel  dcterms:language aat:300387822 ; skosxl:literalForm ?label_literal_en_us . 
      BIND (STRLANG(STR(?label_literal_en_us), 'en') AS  ?label_literal_en)} # lang: @en-US - converts to @en
    UNION
    { ?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . }
}
'''
results_graph = sparql_file_query(query=query__construct_test, format='ttl', filepath='tests/sample-data.ttl')
results_graph.serialize(destination="tests/sample-data-converted.ttl", format="turtle") # save to tests/sample-data-converted.ttl
print(results_graph.serialize(format="turtle").decode('utf-8'))


# issue with AAT term [aat:300430847](https://vocab.getty.edu/aat/300430847)

https://vocab.getty.edu/aat/300430847 (one term used in PAN deposit) is an odd case, where the @en label is **not a value of the** `skosxl:prefLabel` property, only @fr is. The @en label is however the value of `skosxl:altLabel` and `gvp:prefLabelGVP`

```
aat:300430847 a gvp:Concept ;
	skos:inScheme <http://vocab.getty.edu/aat/> ;
	skosxl:prefLabel aat_term:1000799425-fr ;
    skosxl:altLabel aat_term:1000799424-en ; 
    gvp:prefLabelGVP aat_term:1000799424-en .
```
This issue was [reported](https://groups.google.com/g/gettyvocablod/c/nn3ra62X_Sc/m/Pcnl5y1_AwAJ) and since fixed.

In [77]:
#### CONSTRUCT query to create test data test/sample-data2.ttl ####
# difference from similar query in cell 2 is 
# inclusion of gvp:prefLabelGVP

query_construct = '''
CONSTRUCT
{     ?concept a gvp:Concept ;
               skos:inScheme aat: ;
               skosxl:prefLabel ?preflabel ;
               gvp:prefLabelGVP ?gpvlabel .
      
      ?preflabel ?preflabelProp ?preflabelVal .
} 
WHERE 
{
   VALUES ?concept { aat:300189559 aat:300022441 aat:300456698 aat:300430847 } 

   ?concept a gvp:Concept ;
            skosxl:prefLabel ?preflabel ;
            gvp:prefLabelGVP ?gpvlabel .
   ?preflabel ?preflabelProp ?preflabelVal .

}
'''
testq = sparql_endpoint_query(query=query_construct, format='turtle')
with open('tests/sample-data2.ttl', 'wb') as sampledata:
      sampledata.write(testq)

# output: all 4 concepts have @en labels as values of gvp:prefLabelGVP

KeyboardInterrupt: 

In [62]:
### Are the Concepts without gvp:prefLabelGVP property
query_no_labelGVP = '''
SELECT *
WHERE 
{
   ?concept a gvp:Concept .
   MINUS { ?concept gvp:prefLabelGVP ?gpvlabel }.
}
LIMIT 100
'''
testq = sparql_endpoint_query(query=query_no_labelGVP, format='json')
pprint(testq)

# 0 results - hence, it means that all concepts have a  gvp:prefLabelGVP

{'head': {'vars': ['concept', 'gpvlabel']}, 'results': {'bindings': []}}


In [69]:
### All the Concepts & the gvp:prefLabelGVP values to CSV tests/prefLabelGVP.csv
query_no_labelGVP = '''
SELECT *
WHERE 
{
   ?concept a gvp:Concept .
   OPTIONAL {?concept gvp:prefLabelGVP ?gpvlabel } .
}
ORDER BY ?concept
'''
testq = sparql_endpoint_query(query=query_no_labelGVP, format='csv')
with open('tests/prefLabelGVP.csv', 'wb') as sampledata:
      sampledata.write(testq)
# 0 results - hence, it means that all concepts have a  gvp:prefLabelGVP

In [4]:
### All the Concepts & the gvp:prefLabelGVP which do not end in "-en"
# stored in tests/prefLabelGVP-exceptions.csv
query_no_labelGVP = '''
SELECT *
WHERE 
{
   ?concept a gvp:Concept .
   ?concept gvp:prefLabelGVP ?gpvlabel .
   FILTER ( !regex(STR(?gpvlabel), "-en.*$") ) 
   ?gpvlabel skosxl:literalForm ?literal .  
}
ORDER BY ?concept
'''
testq = sparql_endpoint_query(query=query_no_labelGVP, format='csv')
with open('tests/prefLabelGVP-exceptions.csv', 'wb') as sampledata:
      sampledata.write(testq)


## results from gvp:prefLabelGVP exploration

The result from the previous, stored in [tests/prefLabelGVP-exceptions.csv](tests/prefLabelGVP-exceptions.csv), shows that there are come concepts, with  `gvp:prefLabelGVP` values that are not in English, which in some cases makes sense, since they are concepts that are language/culture specific (ie. http://vocab.getty.edu/aat/300457647 dǒukē) and the ones that fall outside that, have been reported.

So this gives me confidence that I can extract the English labels from the `gvp:prefLabelGVP` values, as they tend to be in English.

# Generate AATC w gvp:prefLabelGVP values for EN labels

Requirements: 
* EN labels are captured from the value nodes of the prop `gvp:prefLabelGVP` 
* include Dutch labels (when present)


In [112]:
query_concept_labelGVP = '''
SELECT DISTINCT ?concept ?label_literal_en ?label_literal_en_us ?label_literal_nl ?gpv_label_langNode 
WHERE{
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel ;
        gvp:prefLabelGVP ?gpvlabel .        
    OPTIONAL {?concept dcterms:issued ?issuedDate .}
    FILTER( STRSTARTS(str(?concept), str(aat:)) )
    
    #   ?gpvlabel  dcterms:language aat:300388277 .  # No need as prefLabelGVP values are EN by default
      ?gpvlabel skosxl:literalForm ?label_literal_en ;  
                dcterms:language ?gpv_label_langNode . 
       OPTIONAL {?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . }
    
}
# ORDER BY ?concept
LIMIT 100
'''
responses = sparql_endpoint_query(query=query_concept_labelGVP, format='csv')
with open('tests/prefLabelGVP-EN-NL.csv', 'wb') as sampledata:
      sampledata.write(responses)


# simplify the query by goingfetching the gvplabel in whatever lang it is in
# and fetch the Dutch, if it is there



In [109]:
query_concept_labelGVP = '''
SELECT DISTINCT ?gpv_label_langNode 
WHERE{
    ?concept a gvp:Concept ;
        skos:inScheme aat: ;
        skosxl:prefLabel ?preflabel ;
        gvp:prefLabelGVP ?gpvlabel .        
    OPTIONAL {?concept dcterms:issued ?issuedDate .}

    { 
    #   ?gpvlabel  dcterms:language aat:300388277 .  # redundant 
      ?gpvlabel skosxl:literalForm ?label_literal_en ;  
                dcterms:language ?gpv_label_langNode . 
       OPTIONAL {?preflabel  dcterms:language aat:300388256 ; skosxl:literalForm ?label_literal_nl . }
    }
         FILTER( STRSTARTS(str(?concept), str(aat:)) )
}
# ORDER BY ?concept
LIMIT 10
'''
responses = sparql_endpoint_query(query=query_concept_labelGVP, format='json')
pprint(responses)

# with open('tests/prefLabelGVP-EN-NL.csv', 'wb') as sampledata:
#       sampledata.write(responses)


# simplify the query by goingfetching the gvplabel in whatever lang it is in
# and fetch the Dutch, if it is there





{'head': {'vars': ['gpv_label_langNode']},
 'results': {'bindings': [{'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'http://vocab.getty.edu/aat/300388277'}},
                          {'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'http://vocab.getty.edu/aat/300387822'}},
                          {'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'http://vocab.getty.edu/aat/300388361'}},
                          {'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'http://vocab.getty.edu/aat/300388119'}},
                          {'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'http://vocab.getty.edu/aat/300388256'}},
                          {'gpv_label_langNode': {'type': 'uri',
                                                  'value': 'ht

# Testing aatc.ttl

In [124]:
## HOw manny concepts are there in aatc
q_count_concept = '''
SELECT (COUNT( DISTINCT ?concept) as ?concept_n)
WHERE {
    ?concept a skos:Concept
} 
'''

results_graph = sparql_file_query(query=q_count_concept, format='ttl', filepath='aatc.ttl')
for i in results_graph:
    print(i)
# print(results_graph.serialize(format="turtle").decode('utf-8'))


(rdflib.term.Literal('57085', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')),)


In [ ]:
# are there skos:Concepts outside the aat: namespace ?

q_other_prefixes = '''
SELECT ?concept
WHERE {
    ?concept a skos:Concept .
    FILTER( !STRSTARTS(str(?concept), str(aat:)) )
} 
'''

results_graph = sparql_file_query(query=q_other_prefixes, format='ttl', filepath='aatc.ttl')
for i in results_graph:
    print(i)
# print(results_graph.serialize(format="turtle").decode('utf-8'))


# obsolete concepts

In [127]:
# How many of the AAT concepts are obsolete?

q_count_obsolete = '''
SELECT (COUNT (?obsolete) as ?total)
WHERE {
 ?obsolete a gvp:ObsoleteSubject.
 FILTER( STRSTARTS(str(?obsolete), str(aat:)) ) 
}
'''

responses = sparql_endpoint_query(query=q_count_obsolete, format='json')
pprint(responses)
 




{'head': {'vars': ['total']},
 'results': {'bindings': [{'total': {'datatype': 'http://www.w3.org/2001/XMLSchema#integer',
                                     'type': 'literal',
                                     'value': '1332'}}]}}


In [126]:
# How many of the AAT concepts are obsolete?

q_count_obsolete = '''
SELECT ?obsolete
WHERE {
 ?obsolete a gvp:ObsoleteSubject.
 FILTER( !STRSTARTS(str(?obsolete), str(aat:)) ) 
}
'''

responses = sparql_endpoint_query(query=q_count_obsolete, format='json')
pprint(responses)
 




{'head': {'vars': ['obsolete']},
 'results': {'bindings': [{'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300265851'}},
                          {'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300218074'}},
                          {'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300197033'}},
                          {'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300263277'}},
                          {'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300287378'}},
                          {'obsolete': {'type': 'uri',
                                        'value': 'http://vocab.getty.edu/aat/300288025'}},
                          {'obsolete': {'type': 'uri',
                                    